In [ ]:
# --timeframe 1d   : Таймфрейм свечей (дневные данные).
# --start-year 2000: Глубина загрузки истории (начиная с 2000 года).
# --workers 6      : Количество параллельных потоков для ускорения загрузки.

!python -m _tools.update_market_data --timeframe 1d --start-year 2000 --workers 6
!python -m _tools.update_macro --timeframe 1d --start-year 2000 --workers 6
# Выполняет комплексную проверку целостности, отсутствия пропусков и корректности OHLCV данных.
!python -m _tools.check_data_quality

In [ ]:
# --timeframe 1d       : Интервал данных — дневные свечи.
# --lookback 60        : Глубина истории — модель смотрит на 60 дней назад.
# --horizon 10         : Горизонт прогноза — ищем выход по барьерам в течение 10 дней.
# --auto               : Режим автоматического расчета уровней TP/SL на основе волатильности.
# --percentile 75      : Перцентиль волатильности для отсечения аномальных выбросов при авто-разметке.
# --init_split         : Дата начала первого разделения данных на Train и Val.
# --val_interval 2     : Продолжительность валидационного периода в годах.
# --split_interval 2   : Шаг смещения окна Walk-Forward в годах.
# --endpoint           : Дата окончания формирования всех временных интервалов.
# --corr_threshold     : Порог удаления коррелирующих признаков (убираем дубликаты > 85%).
# --cum_threshold      : Порог кумулятивной важности (оставляем топ фичей, дающих 99% влияния).
# --force              : Раскомментируйте параметр ниже для полной перезаписи кэшированных данных.

!python -m _tools.init_dataset \
    --timeframe 1d \ 
    --lookback 60 \
    --horizon 10 \
    --auto \
    --percentile 75 \
    --init_split 2010-01-01 \
    --val_interval 2 \
    --split_interval 2 \
    --endpoint 2024-01-01 \
    --corr_threshold 0.85 \
    --cum_threshold 0.99 \
    #--force


🧹 Запуск модуля очистки данных (Сплиты, Иглы, Выбросы)...
Корректировка сплитов:  10%|██▏                  | 7/68 [00:00<00:00, 61.70it/s]  🕵️‍♂️ [HEURISTIC SPLIT] LSNGP@MISX на 2005-08-03: Коэфф 5.0
  📌 [KNOWN SPLIT] TRNFP@MISX на 2024-02-21: Коэфф 0.01
Корректировка сплитов: 100%|████████████████████| 68/68 [00:00<00:00, 78.60it/s]
✅ Очистка завершена!

🔄 [fold_2010/train - Build] Запуск...
✅ Готово: сохранено в /home/restorator/trader_test/data/processed/2000_2026_1d/fold_2010/data/train/dataset.csv

🔄 [fold_2010/train - Labels] Запуск...
✅ Авто-уровни: TP=11.68%, SL=10.05%
Разметка: 100%|████████████████████████████████| 39/39 [00:00<00:00, 178.53it/s]
🎉 Размеченный датасет сохранен: labels.csv

🔄 [fold_2010/train - Features] Запуск...

⚙️ [TRAIN] Инициализация расчета...
Этап 1: Расчет кросс-секционных признаков...
Индивидуальные фичи: 100%|██████████████████████| 33/33 [00:43<00:00,  1.32s/it]
✂️ Winsorization (0.1% - 99.9%) для 149 признаков...
📈 Обучение Scaler...
💾 Сохранено 

In [10]:
!python -m _tools.verify_data

✅ Аудит завершен. Отчет обновлен и сохранен: /home/restorator/trader_test/data_audit_report.txt


In [17]:
!rm -rf data/processed

In [ ]:
!python -m _tools.train_model \
    --exp_name exp_30_5_1d \
    --batch_size 16384 \
    #--batch_size 8192 \
    --epochs 50 \
    --runs 200

In [ ]:
!python -m _tools.prepare_rl_env --exp_fast exp_30_5_1d --exp_slow exp_60_10_1d

In [ ]:
!python -m _tools.train_rllib_pbt --population 4

In [ ]:
!python -m _tools.evaluate